# Support vector machines

Logistic regression finds a line that separates the classes. Infinitely many lines can do that, so which one is best? The support vector machine picks the one with the widest empty margin on either side. Add the kernel trick and the same idea draws curved boundaries.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from common.datasets import load_breast_cancer_data, load_digits_data, load_iris_flowers
from common.plotting import plot_decision_boundary

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

Picture two groups of points and a road laid between them. You want the widest road that keeps every point on its own side. The edges of the road touch a few points. Those are the **support vectors**, and they alone decide where the road goes. Move any other point (without crossing the road) and nothing changes.

A wide margin is a bet on generalisation. A boundary hugging the training points is fragile, because a slightly different sample could tip a point over it. A boundary with room on both sides tolerates that.

Real data overlaps, so no road is clear. The **soft margin** allows some points inside the road or on the wrong side, at a price. The parameter $C$ sets that price.

## 2. Maths

Use labels $y_i\in\{-1,+1\}$ and a score $f(\mathbf x)=\mathbf w^\top\mathbf x+b$. The margin of a point is $y_i f(\mathbf x_i)$: positive when correct, and larger when further from the line. The road's total width is $2/\lVert\mathbf w\rVert$, so a wide road means a small $\lVert\mathbf w\rVert$.

**Hard margin.** Minimise $\tfrac12\lVert\mathbf w\rVert^2$ subject to $y_i f(\mathbf x_i)\ge1$ for all $i$. It only has a solution when the classes are perfectly separable.

**Soft margin and the hinge loss.** Let each point have slack $\xi_i\ge0$, and the cost becomes

$$J=\tfrac12\lVert\mathbf w\rVert^2+C\sum_i\max\bigl(0,\;1-y_if(\mathbf x_i)\bigr)$$

The second term is the **hinge loss**. It is zero for any point with margin at least 1 (correct and outside the road), and grows linearly as the point moves inside the road or across the line. Compare it with log-loss from notebook 20: the logistic loss is never exactly zero and keeps pushing, while the hinge loss ignores well-classified points entirely. That is why only the support vectors matter.

$C$ multiplies the data term, so **large $C$ means a narrow road and few violations** (low bias, high variance) and **small $C$ means a wide road and more violations** (regularisation). Dividing through by $C$ gives the same shape as an L2-penalised model with $\lambda=1/C$, and $C$ here matches `C` in `LogisticRegression`.

**Kernels.** The training problem can be written so that data only enter as dot products $\mathbf x_i\cdot\mathbf x_j$. Swap each dot product for a kernel $K(\mathbf x_i,\mathbf x_j)$, which equals a dot product in some richer feature space, and you get a curved boundary in the original space without ever computing the richer features. The usual choice is the **RBF kernel**:

$$K(\mathbf a,\mathbf b)=\exp\bigl(-\gamma\lVert\mathbf a-\mathbf b\rVert^2\bigr)$$

It is a similarity that is 1 for identical points and decays with distance. $\gamma$ sets how fast. Large $\gamma$ means each point's influence is very local, giving a wiggly, high-variance boundary. Small $\gamma$ means smooth and nearly linear.

## 3. From scratch: hinge loss with gradient descent

A linear SVM is a few lines. The subgradient of the hinge term is $-y_i\mathbf x_i$ for points with margin under 1, and zero otherwise. I use two iris features (setosa against the rest would be too easy, so versicolor against virginica) and standardise them.

In [ ]:
X_iris, y_iris = load_iris_flowers()
mask = y_iris.isin([1, 2])
X2 = X_iris.loc[mask, ["petal_length", "petal_width"]].to_numpy()
y2 = (y_iris[mask] == 2).astype(int).to_numpy()
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.3, random_state=SEED, stratify=y2)
sc = StandardScaler().fit(X2_tr)
Xs_tr, Xs_te = sc.transform(X2_tr), sc.transform(X2_te)


def fit_linear_svm(
    X: np.ndarray, y01: np.ndarray, C: float = 1.0, lr: float = 0.01, steps: int = 3000
) -> tuple[np.ndarray, float, list[float]]:
    y = 2 * y01 - 1  # labels in {-1, +1}
    w, b, history = np.zeros(X.shape[1]), 0.0, []
    for _ in range(steps):
        margin = y * (X @ w + b)
        viol = margin < 1
        history.append(0.5 * w @ w + C * np.maximum(0, 1 - margin).sum())
        w -= lr * (w - C * (y[viol, None] * X[viol]).sum(axis=0))
        b -= lr * (-C * y[viol].sum())
    return w, b, history


w, b, history = fit_linear_svm(Xs_tr, y2_tr, C=1.0)
ref = SVC(kernel="linear", C=1.0).fit(Xs_tr, y2_tr)
print("scratch:", w, round(b, 3), " cost:", round(history[-1], 3))
print("sklearn:", ref.coef_[0], round(ref.intercept_[0], 3))
print("support vectors:", ref.n_support_.sum(), "of", len(y2_tr))

Both land on nearly the same line, and only 12 of the 70 points are support vectors. Everything else could be deleted without changing the answer.

## 4. Soft margin and C

Plot the linear SVM on the two-species problem at three values of $C$, marking the support vectors and the margin edges (where the score is $\pm1$).

In [ ]:
def plot_margin(model: SVC, X: np.ndarray, y: np.ndarray, ax: plt.Axes) -> None:
    gx, gy = np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-2.5, 2.5, 200))
    z = model.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, z > 0, alpha=0.2, cmap="viridis")
    ax.contour(gx, gy, z, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"], linewidths=1)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", edgecolor="k", s=25)
    ax.scatter(*model.support_vectors_.T, s=90, facecolors="none", edgecolors="r")


fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, C in zip(axes, (0.01, 1, 100)):
    m = SVC(kernel="linear", C=C).fit(Xs_tr, y2_tr)
    plot_margin(m, Xs_tr, y2_tr, ax)
    ax.set_title(f"C={C}: {m.n_support_.sum()} support vectors, test {m.score(Xs_te, y2_te):.2f}")
plt.show()

At $C=0.01$ the road is wide and most points are support vectors, because most sit inside it. At $C=100$ the road is narrow and the boundary fits the training points closely. Both are equally good on test data here, because the problem is easy. The trade matters more when classes overlap heavily or the data are noisy.

## 5. Kernels: curved boundaries

A straight line cannot separate two interleaved half-moons. The RBF kernel can. Use the `make_moons` toy set with a little noise, and vary $\gamma$ with $C$ fixed.

In [ ]:
Xm, ym = make_moons(n_samples=300, noise=0.2, random_state=SEED)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.4, random_state=SEED)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
setups = [
    ("linear", {}),
    ("rbf, gamma=0.1", {"gamma": 0.1}),
    ("rbf, gamma=1", {"gamma": 1}),
    ("rbf, gamma=200", {"gamma": 200}),
]
for ax, (name, kw) in zip(axes, setups):
    m = SVC(kernel=name.split(",")[0], C=1, **kw).fit(Xm_tr, ym_tr)
    plot_decision_boundary(m, Xm_tr, ym_tr, ax, labels=("x1", "x2"))
    ax.set_title(
        f"{name}\ntrain {m.score(Xm_tr, ym_tr):.2f}, test {m.score(Xm_te, ym_te):.2f}", fontsize=9
    )
plt.show()

The linear model cuts through the moons and scores 0.875 on test data. $\gamma=0.1$ is smooth and nearly linear (underfit). $\gamma=1$ follows the shape and is the best at 0.97. $\gamma=200$ draws a little island around every training point: training accuracy 0.99 and test accuracy down to 0.83. That is the same overfitting we met in notebook 13, controlled by $\gamma$ instead of polynomial degree.

## 6. Tuning C and gamma

$C$ and $\gamma$ interact: a large $\gamma$ needs a smaller $C$ to calm it, and vice versa. Search them together on a log grid, with scaling inside the pipeline. Breast cancer is the test case, with 30 features of very different scales.

In [ ]:
Xb, yb = load_breast_cancer_data()
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.3, random_state=SEED, stratify=yb)
pipe = make_pipeline(StandardScaler(), SVC())
params = {"svc__C": np.logspace(-2, 3, 6), "svc__gamma": np.logspace(-5, 0, 6)}
search = GridSearchCV(pipe, params, cv=5).fit(Xb_tr, yb_tr)
print("best:", search.best_params_, " CV accuracy:", round(search.best_score_, 4))
print("test accuracy:", round(search.score(Xb_te, yb_te), 4))
table = search.cv_results_["mean_test_score"].reshape(6, 6)
fig, ax = plt.subplots(figsize=(5.5, 4))
im = ax.imshow(table, cmap="viridis")
ax.set_xticks(range(6), [f"{g:.0e}" for g in params["svc__gamma"]])
ax.set_yticks(range(6), [f"{c:.0e}" for c in params["svc__C"]])
ax.set_xlabel("gamma")
ax.set_ylabel("C")
plt.colorbar(im)
plt.show()

The good region is a diagonal band: more of one needs less of the other. Too small a $\gamma$ with a small $C$ underfits, and a large $\gamma$ drops accuracy in every row because the model memorises. Use a coarse log grid first, then refine around the best cell. A linear kernel is also worth a try on data with many features (`LinearSVC` is far faster on large datasets).

**Cost reduction.** Training an exact kernel SVM costs between $O(n^2)$ and $O(n^3)$, so it gets slow past tens of thousands of rows. `LinearSVC` or `SGDClassifier(loss="hinge")` scale to millions. Subsample while tuning.

## 7. What breaks

**Scaling.** The RBF kernel uses distances, so unscaled features break it as they broke kNN.

In [ ]:
raw = SVC(C=10, gamma="scale")
scaled = make_pipeline(StandardScaler(), SVC(C=10, gamma="scale"))
print("unscaled:", cross_val_score(raw, Xb, yb, cv=5).mean().round(4))
print("scaled:  ", cross_val_score(scaled, Xb, yb, cv=5).mean().round(4))

**No probabilities.** The SVM outputs a signed distance to the line, not a probability. `SVC(probability=True)` fits a sigmoid on top by cross-validation, which is slower and not always well calibrated (notebook 25).

**Multiclass.** SVMs are binary. sklearn handles ten digits by training one classifier per pair of classes and voting.

In [ ]:
Xd, yd = load_digits_data()
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, random_state=SEED, stratify=yd)
for gamma in ("scale", 1, 10):
    m = SVC(C=10, gamma=gamma).fit(Xd_tr / 16, yd_tr)
    print(
        f"digits, gamma={gamma}: train {m.score(Xd_tr / 16, yd_tr):.3f}  test {m.score(Xd_te / 16, yd_te):.3f}"
    )

With the default `gamma="scale"` (one over the number of features times the feature variance) the SVM reaches about 99% on digits, level with kNN. Raising $\gamma$ to 1 costs a little, and at 10 the model memorises the training set (train 1.000) and test accuracy collapses to 0.16, below what guessing the most common digit would give.

**Other limits.** Hard to interpret, slow on large data, and sensitive to the $C$ and $\gamma$ you choose. Strong when there are few hundred to few thousand rows with many features.

Next, a model that is easy to read: a tree of questions.